# Phase 3 â€” Model-Based CF: Matrix Factorization (PyTorch)
Learns latent user and item embeddings to predict ratings. Trains with MSE loss + Adam + L2 regularization.

In [ ]:
import sys, time
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.cf import MatrixFactorization
from src.data import load_movies

In [ ]:
train  = pd.read_csv('../data/train.csv')
val    = pd.read_csv('../data/val.csv')
test   = pd.read_csv('../data/test.csv')
movies = load_movies()
print(f'train: {train.shape}  val: {val.shape}  test: {test.shape}')

## 1. Train Matrix Factorization
- : dimensionality of user/item latent vectors
- : full passes over training data
- , : Adam learning rate; L2 applied to embeddings only in loss
- : mini-batch SGD
-  is baked into the model so predictions start near the average rating (~3.58)

In [ ]:
t0 = time.time()
mf = MatrixFactorization(n_factors=64, n_epochs=20, lr=0.005, reg=0.1, batch_size=2048)
mf.fit(train, val)
print(f'
Total training time: {time.time()-t0:.1f}s')
best_epoch = int(np.argmin(mf.history["val_loss"])) + 1
print(f'Best val RMSE at epoch {best_epoch}: {min(mf.history["val_loss"]):.4f}')

## 2. Training Curve

In [ ]:
epochs = range(1, len(mf.history['train_loss']) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, mf.history['train_loss'], label='Train RMSE')
plt.plot(epochs, mf.history['val_loss'],   label='Val RMSE')
plt.xlabel('Epoch')
plt.ylabel('RMSE')
plt.title('Matrix Factorization â€” Learning Curve')
plt.legend()
plt.tight_layout()
plt.savefig('../data/mf_learning_curve.png', dpi=120)
plt.show()
print(f'Final train RMSE: {mf.history["train_loss"][-1]:.4f}')
print(f'Final val   RMSE: {mf.history["val_loss"][-1]:.4f}')

## 3. Recommendations for Sample User

In [ ]:
USER_ID = 1
recs = mf.recommend(user_id=USER_ID, n=10)
rec_df = pd.DataFrame(recs, columns=['movieId', 'score'])
rec_df = rec_df.merge(movies[['movieId', 'title', 'genres']], on='movieId')
print(f'MF recommendations for user {USER_ID}:')
rec_df

## 4. Precision@10 on Validation Set

In [ ]:
def precision_at_k(model, val_df, k=10, n_users=200):
    val_items = val_df.groupby('userId')['movieId'].apply(set).to_dict()
    sample_users = list(val_items.keys())[:n_users]
    hits = []
    for uid in sample_users:
        recs = model.recommend(uid, n=k)
        rec_ids = {r[0] for r in recs}
        hits.append(len(rec_ids & val_items[uid]) / k)
    return np.mean(hits)

p_at_10 = precision_at_k(mf, val, k=10)
print(f'MF  Precision@10 (first 200 users): {p_at_10:.4f}')

## 5. Summary

| Model | Precision@10 | Val RMSE |
|---|---|---|
| User-User CF | 0.0015 | — |
| Item-Item CF | 0.0855 | — |
| Matrix Factorization | 0.0070 | 0.8822 (best at epoch 2) |

MF learns meaningful latent factors (val RMSE 0.88 vs random baseline ~3.5), but overfits after epoch 2 — train RMSE reaches 0.51 while val RMSE climbs to 1.12. Item-Item CF still wins on Precision@10 because the overfitted MF model surfaces obscure high-score items rather than items the user will actually watch next. Early stopping at epoch 2 would give better ranking quality.